In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv('for_model_selection_train_df.csv').drop(columns=['store_room','floor_category','balcony'])

In [4]:
df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,pooja_room,study_room,servant_room,others,furnishing_type,luxury_category,NearbyLocationsCategory
0,0,flat,sohna road,0.35,2,2,Relatively New,556.0,0,0,0,0,0,Low,Low
1,1,flat,sector 56,2.05,4,4,Moderately Old,2667.0,0,0,0,0,0,Low,High
2,2,flat,sector 90,1.25,3,3,Relatively New,1700.0,1,0,1,0,0,Medium,Low
3,3,flat,sector 56,1.45,3,3,Old Property,1900.0,1,1,1,1,0,High,High
4,4,flat,sector 92,0.66,3,2,Relatively New,1473.0,0,0,0,0,1,Low,Low


In [5]:
df = df.drop(columns=['Unnamed: 0','pooja_room','others'])

In [6]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,study_room,servant_room,furnishing_type,luxury_category,NearbyLocationsCategory
0,flat,sohna road,0.35,2,2,Relatively New,556.0,0,0,0,Low,Low
1,flat,sector 56,2.05,4,4,Moderately Old,2667.0,0,0,0,Low,High
2,flat,sector 90,1.25,3,3,Relatively New,1700.0,0,1,0,Medium,Low
3,flat,sector 56,1.45,3,3,Old Property,1900.0,1,1,0,High,High
4,flat,sector 92,0.66,3,2,Relatively New,1473.0,0,0,1,Low,Low


In [7]:
# 0 -> unfurnished
# 1 -> furnished

In [8]:
# numerical -> bedRoom,bathroom,built_up_area,servant_room,study_room
# ordinal -> property_type,furnishing_type,luxury_category,NearbyLocationsCategory
# OHE -> sector , agePossession

In [9]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [10]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,study_room,servant_room,furnishing_type,luxury_category,NearbyLocationsCategory
0,flat,sohna road,0.35,2,2,new,556.0,0,0,0,Low,Low
1,flat,sector 56,2.05,4,4,old,2667.0,0,0,0,Low,High
2,flat,sector 90,1.25,3,3,new,1700.0,0,1,0,Medium,Low
3,flat,sector 56,1.45,3,3,old,1900.0,1,1,0,High,High
4,flat,sector 92,0.66,3,2,new,1473.0,0,0,1,Low,Low


In [11]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\THARUN TEJA S\AppData\Local\Temp\ipykernel_28904\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [12]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,study_room,servant_room,furnishing_type,luxury_category,NearbyLocationsCategory
0,0,sohna road,0.35,2,2,new,556.0,0,0,0,Low,Low
1,0,sector 56,2.05,4,4,old,2667.0,0,0,0,Low,High
2,0,sector 90,1.25,3,3,new,1700.0,0,1,0,Medium,Low
3,0,sector 56,1.45,3,3,old,1900.0,1,1,0,High,High
4,0,sector 92,0.66,3,2,new,1473.0,0,0,1,Low,Low


In [13]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})
df.head()

C:\Users\THARUN TEJA S\AppData\Local\Temp\ipykernel_28904\3270725456.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})


,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,study_room,servant_room,furnishing_type,luxury_category,NearbyLocationsCategory
0,0,sohna road,0.35,2,2,new,556.0,0,0,0,0,Low
1,0,sector 56,2.05,4,4,old,2667.0,0,0,0,0,High
2,0,sector 90,1.25,3,3,new,1700.0,0,1,0,1,Low
3,0,sector 56,1.45,3,3,old,1900.0,1,1,0,2,High
4,0,sector 92,0.66,3,2,new,1473.0,0,0,1,0,Low


In [14]:
df['NearbyLocationsCategory'] = df['NearbyLocationsCategory'].replace({'Low':0,'Medium':1,'High':2})
df.head()

C:\Users\THARUN TEJA S\AppData\Local\Temp\ipykernel_28904\1081589903.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['NearbyLocationsCategory'] = df['NearbyLocationsCategory'].replace({'Low':0,'Medium':1,'High':2})


,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,study_room,servant_room,furnishing_type,luxury_category,NearbyLocationsCategory
0,0,sohna road,0.35,2,2,new,556.0,0,0,0,0,0
1,0,sector 56,2.05,4,4,old,2667.0,0,0,0,0,2
2,0,sector 90,1.25,3,3,new,1700.0,0,1,0,1,0
3,0,sector 56,1.45,3,3,old,1900.0,1,1,0,2,2
4,0,sector 92,0.66,3,2,new,1473.0,0,0,1,0,0


In [15]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [38]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [39]:
y_log = np.log1p(y)

In [40]:
y_log

0       0.300105
1       1.115142
2       0.810930
3       0.896088
4       0.506818
          ...   
3549    1.011601
3550    2.250239
3551    0.371564
3552    1.408545
3553    1.545433
Name: price, Length: 3554, dtype: float64

In [41]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [42]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [43]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,study_room,servant_room,furnishing_type,luxury_category,NearbyLocationsCategory,sector_gwal pahari,...,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_old,agePossession_under construction
0,-0.515431,-0.872458,-0.874873,-1.070215,-0.480218,-0.747510,-0.323130,-0.986678,-1.287019,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,4.730917,-0.05572,-0.604874,-0.191719
1,-0.515431,0.734546,0.507037,0.658403,-0.480218,-0.747510,-0.323130,-0.986678,1.484302,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,1.653237,-0.191719
2,-0.515431,-0.068956,-0.183918,-0.133437,-0.480218,1.337774,-0.323130,0.439371,-1.287019,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.604874,-0.191719
3,-0.515431,-0.068956,-0.183918,0.030336,2.082387,1.337774,-0.323130,1.865419,1.484302,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,1.653237,-0.191719
4,-0.515431,-0.068956,-0.874873,-0.319318,-0.480218,-0.747510,3.094734,-0.986678,-1.287019,-0.071348,...,5.877074,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.604874,-0.191719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,-0.515431,-0.068956,0.507037,0.158078,-0.480218,-0.747510,-0.323130,1.865419,0.098642,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.604874,-0.191719
3550,1.940122,-0.068956,-0.183918,0.685425,-0.480218,1.337774,-0.323130,-0.986678,1.484302,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.604874,-0.191719
3551,-0.515431,-0.872458,-0.874873,-0.752497,-0.480218,-0.747510,-0.323130,-0.986678,1.484302,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.604874,-0.191719
3552,1.940122,1.538048,1.197993,1.465801,-0.480218,1.337774,-0.323130,-0.986678,-1.287019,-0.071348,...,-0.170153,-0.050386,-0.125375,-0.058206,-0.092266,-0.053119,-0.211375,-0.05572,-0.604874,-0.191719


In [44]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [45]:
scores.mean(),scores.std()

(np.float64(0.846895594560489), np.float64(0.01796802654984921))

In [46]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [47]:
lr.fit(X_scaled,y_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [48]:
ridge.fit(X_scaled,y_log)


,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",Non

In [49]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,123),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [50]:
coef_df

,feature,coef
0,property_type,0.118119
1,bedRoom,0.054013
2,bathroom,0.071012
3,built_up_area,0.196661
4,study_room,0.019118
...,...,...
118,sector_sector 9a,-0.006750
119,sector_sohna road,-0.033423
120,sector_sohna road road,-0.010726
121,agePossession_old,-0.004194


In [52]:
coef_df = pd.DataFrame(lr.coef_.reshape(1,123),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [53]:
coef_df

,feature,coef
0,property_type,0.118119
1,bedRoom,0.054013
2,bathroom,0.071012
3,built_up_area,0.196661
4,study_room,0.019118
...,...,...
118,sector_sector 9a,-0.006750
119,sector_sohna road,-0.033423
120,sector_sohna road road,-0.010726
121,agePossession_old,-0.004194


In [54]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(Ridge(alpha=0.0001), X_scaled, y_log, cv=kfold, scoring='r2')

In [55]:
scores.mean(),scores.std()

(np.float64(0.8468955967288234), np.float64(0.017968027766989303))

In [56]:
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.861
Model:                            OLS   Adj. R-squared:                  0.856
Method:                 Least Squares   F-statistic:                     173.0
Date:                Thu, 12 Nov 2026   Prob (F-statistic):               0.00
Time:                        20:17:04   Log-Likelihood:                 549.26
No. Observations:                3554   AIC:                            -850.5
Df Residuals:                    3430   BIC:                            -84.71
Df Model:                         123                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [57]:
y_log.std()

np.float64(0.5565373265235526)

In [58]:
X_scaled['bedRoom'].std()

np.float64(1.000140716246387)

In [59]:
0.054013 *(0.5565/1.0001)

0.03005522897710229

In [60]:
np.expm1(0.03005522897710229)

np.float64(0.030511446474973232)

In [61]:
0.030511446474973232*100

3.051144647497323